In [343]:
import warnings
warnings.filterwarnings(
    "ignore",
    message=r"pkg_resources is deprecated as an API\..*",
    category=UserWarning,
)

In [344]:
from src.helpers.model_matrix import build_model_matrix_from_wrds
from src.helpers.model_matrix import align_and_fill_dates_across_tickers

tickers_list = [
    'AAPL', 'NVDA', 'MSFT', 'AMZN', 'TSLA',
    'GOOGL', 'LLY', 'WMT', 'JPM', 'BRK-B', 
    'V', 'MA', 'XOM', 'ORCL', 'UNH', 'COST', 'PG', 'HD', 'NFLX',
    'JNJ', 'BAC', 'CRM', 'QQQ', 'ABBV', 'KO', 'CVX', 'TMUS', 'MRK', 'CSCO',
    'WFC', 'ACN', 'NOW', 'TSM', 'AXP', 'PEP', 'MCD', 'IBM', 'MS', 'DIS',
    'TMO', 'ABT', 'AMD', 'ADBE', 'PM', 'ISRG', 'GE', 'GS', 'INTU', 'CAT',
    'TXN', 'QCOM', 'RY', 'VZ', 'DHR', 'BKNG', 'T', 'BLK', 'SPGI',
    'RTX', 'PFE', 'NEE', 'HON', 'CMCSA', 'PGR', 'AMGN', 'LOW', 'ANET', 'UNP',
    'SYK', 'TJX', 'C', 'BA', 'SCHW', 'BSX', 'KKR', 'ETN',
    'COP', 'BX', 'PANW', 'ADP'
]

all_stocks = build_model_matrix_from_wrds(
    wrds_user="jbernatchez",
    start="2016-01-01",
    end="2021-01-01",
    chunk_size=500_000,
    tickers=tickers_list,
    use_run="run_20250930_202918"  # "new", "last", or a specific folder name (i.e. "run_20250914_133747")
)

df = align_and_fill_dates_across_tickers(all_stocks=all_stocks)

assert df.index.names == ["permno", "date"], "df must be a MultiIndex ['permno','date']"

[info] Using run folder: run_20250930_202918 (reuse=True)
[info] Reuse mode: all required Parquet files are present. No extraction performed.
{'run_folder': 'wrds_extracts\\run_20250930_202918', 'reuse': True, 'artifacts': {'dsf.parquet': 'wrds_extracts\\run_20250930_202918\\dsf.parquet', 'stocknames.parquet': 'wrds_extracts\\run_20250930_202918\\stocknames.parquet', 'ff.parquet': 'wrds_extracts\\run_20250930_202918\\ff.parquet', 'ibes_stats.parquet': 'wrds_extracts\\run_20250930_202918\\ibes_stats.parquet', 'ibes_act.parquet': 'wrds_extracts\\run_20250930_202918\\ibes_act.parquet'}}
[info] Removed 0 permnos(companies) for having zero in cfacpr or cfacshr
[info] Removed 0 permnos(companies) for exceeding the threshold of negative prices
[info] ibes_stats: 274,942 (official_ticker, stat_date) pairs have >1 row (multiple horizons/periodicities). Will collapse before join.
[info] df_prices(+ibes): 95.3% missing in n_analysts.
[info] df_prices(+ibes): 95.3% missing in cons_mean.
[info] df_

In [345]:
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

random.seed(42)

print("Train/Validation/Test Split Configuration")


# Get all unique dates
dates_all = df.index.get_level_values("date").unique().sort_values()
D = len(dates_all)  # Total number of trading dates
N_rows = df.shape[0]  # Total rows

print("\nTotal Data:")
print(f"  Dates: {D} trading days")
print(f"  Period: {dates_all.min().date()} to {dates_all.max().date()}")
print(f"  Rows: {N_rows:,} (stocks × dates)")


# SPLIT: IN-SAMPLE (80%) vs OUT-OF-SAMPLE (20%)


# Split point: 80% for development, 20% for final testing
SPLIT_PCT = 0.80  # First 80% = in-sample, last 20% = out-of-sample
split_idx = int(D * SPLIT_PCT)

# IN-SAMPLE: Used for feature selection, hyperparameter tuning, model development
dates_insample = dates_all[:split_idx]

# OUT-OF-SAMPLE: NEVER touched until final evaluation
dates_outofsample = dates_all[split_idx:]

# Split date (boundary between in-sample and out-of-sample)
SPLIT_DATE = dates_outofsample[0]


print("\nData Split:")
print("   In-Sample (Development Set):")
print(f"   Period: {dates_insample.min().date()} to {dates_insample.max().date()}")
print(f"   Dates: {len(dates_insample)} days ({len(dates_insample)/D*100:.1f}%)")
print("   Purpose: Feature selection, hyperparameter tuning, rolling CV")
print("   Status: Can be used for model development")

print("\nOut-Of-Sample (Test Set):")
print(f"   Period: {dates_outofsample.min().date()} to {dates_outofsample.max().date()}")
print(f"   Dates: {len(dates_outofsample)} days ({len(dates_outofsample)/D*100:.1f}%)")
print("   Purpose: FINAL performance evaluation ONLY")
print("   Status: NEVER seen during training!")

print(f"\nSplit Date: {SPLIT_DATE.date()}")

# Rolling Window Parameters (For In-Sample Only)

fixedWindow = True  # True = sliding window, False = expanding window

# Calculate window sizes based on IN-SAMPLE data only
D_insample = len(dates_insample)


# WINDOW SIZE CONFIGURATION (60/20/20 Split)

# Training window: 60% of in-sample (provides stable model estimates)
initial = int(0.6 * D_insample)

# Validation window: 20% of in-sample (enough to evaluate strategy)
horizon = int(0.2 * D_insample)

# Optimal Number of Folds (Controls step size)

# Target number of rolling windows (folds)
# - Research suggests 5-10 folds is optimal for financial time-series CV
# - Too few (<5): Weak cross-validation, high variance in estimates
# - Too many (>15): Diminishing returns, excessive computation time
# - Recommended: 8-10 folds for good balance

target_folds = 10  # Adjust this value (5-10 recommended)

# Calculate step size to achieve target number of folds
# Formula: step = (remaining_data) / (target_folds - 1)
remaining_data = D_insample - initial - horizon
step = max(1, remaining_data // max(1, target_folds - 1))

# Calculate actual number of windows
actual_folds = (D_insample - initial - horizon) // step + 1

# Ensure windows fit within in-sample data
if initial + horizon > D_insample:
    # Fallback for small datasets
    initial = max(60, int(0.5 * D_insample))
    horizon = max(20, int(0.15 * D_insample))
    remaining_data = D_insample - initial - horizon
    step = max(1, remaining_data // max(1, target_folds - 1))
    actual_folds = (D_insample - initial - horizon) // step + 1
    print("  Warning: Limited data, adjusted window sizes")

print("\nRolling Window Configuration (In-Sample Only):")
print(f"   Training window: {initial} days (~{initial/252:.1f} years, {initial/D_insample*100:.1f}% of in-sample)")
print(f"   Validation window: {horizon} days (~{horizon/252:.1f} years, {horizon/D_insample*100:.1f}% of in-sample)")
print(f"   Step size: {step} days (~{step/5:.1f} weeks)")
print(f"   Target folds: {target_folds}")
print(f"   Actual folds: {actual_folds}")
print(f"   Coverage: {actual_folds * horizon} validation days out of {D_insample} total")
print(f"   Window type: {'Fixed (sliding)' if fixedWindow else 'Expanding'}")
print("\n Tip: Adjust 'target_folds' above to control # of windows (5-15 recommended)")

Train/Validation/Test Split Configuration

Total Data:
  Dates: 696 trading days
  Period: 2018-03-28 to 2020-12-30
  Rows: 54,984 (stocks × dates)

Data Split:
   In-Sample (Development Set):
   Period: 2018-03-28 to 2020-06-11
   Dates: 556 days (79.9%)
   Purpose: Feature selection, hyperparameter tuning, rolling CV
   Status: Can be used for model development

Out-Of-Sample (Test Set):
   Period: 2020-06-12 to 2020-12-30
   Dates: 140 days (20.1%)
   Purpose: FINAL performance evaluation ONLY
   Status: NEVER seen during training!

Split Date: 2020-06-12

Rolling Window Configuration (In-Sample Only):
   Training window: 333 days (~1.3 years, 59.9% of in-sample)
   Validation window: 111 days (~0.4 years, 20.0% of in-sample)
   Step size: 12 days (~2.4 weeks)
   Target folds: 10
   Actual folds: 10
   Coverage: 1110 validation days out of 556 total
   Window type: Fixed (sliding)

 Tip: Adjust 'target_folds' above to control # of windows (5-15 recommended)


In [346]:
# Binary Target Definition
# adjclose_lead = next-day log return(t -> t+1)
# We predict: Will the stock go UP (1) or DOWN (0) tomorrow?

DIR_binary = (df["adjclose_lead"] > 0).astype(int)  # 1 = Up, 0 = Down

# Check class balance (should be roughly 50/50 for market neutrality)
print("Binary Target Distribution")
print(f"\n Up (1):   {(DIR_binary == 1).sum():,} observations ({(DIR_binary == 1).mean()*100:.1f}%)")
print(f" Down (0): {(DIR_binary == 0).sum():,} observations ({(DIR_binary == 0).mean()*100:.1f}%)")
print(f" Total:    {len(DIR_binary):,} observations")

# Feature columns: everything except ticker and target
num_cols = [c for c in df.columns if c not in (["ticker", "adjclose_lead"])]
print(f"\nUsing {len(num_cols)} features for prediction")

Binary Target Distribution

 Up (1):   29,330 observations (53.3%)
 Down (0): 25,654 observations (46.7%)
 Total:    54,984 observations

Using 39 features for prediction


In [347]:
# Hyperparameter Configuration: Using Defaults
print("ElasticNet Hyperparameters: Literature-Based Defaults")

# Standard Default Values (Most Commonly Used)

# C: Inverse regularization strength
# - Default: 0.1 (moderate regularization)
# - This is the most common value in financial ML literature (Hastie et al., 2009)
# - Balances between preventing overfitting and retaining important features
ELASTICNET_C = 0.1

# l1_ratio: Mix of L1 (Lasso) and L2 (Ridge)
# - Default: 0.7 (70% L1, 30% L2)
# - L1 component does feature selection (zeros out weak features)
# - L2 component handles multicollinearity (keeps correlated features stable)
# - 0.7 is standard for financial applications (Zou & Hastie, 2005)
ELASTICNET_L1_RATIO = 0.7

print("\nHyperparameters:")
print(f"  C = {ELASTICNET_C}")
print("    (Moderate regularization - Hastie et al., 2009)")
print(f"  l1_ratio = {ELASTICNET_L1_RATIO}")
print("    (70% L1 / 30% L2 - Zou & Hastie, 2005)")

print("\nExpected behavior:")
print("  - Will select ~40-50% of features (sparse model)")
print("  - Handles multicollinearity well (L2 component)")
print("  - Standard choice in financial ML applications")

print("  Using defaults (no grid search needed)")
print("  Default values are well-validated in literature")


ElasticNet Hyperparameters: Literature-Based Defaults

Hyperparameters:
  C = 0.1
    (Moderate regularization - Hastie et al., 2009)
  l1_ratio = 0.7
    (70% L1 / 30% L2 - Zou & Hastie, 2005)

Expected behavior:
  - Will select ~40-50% of features (sparse model)
  - Handles multicollinearity well (L2 component)
  - Standard choice in financial ML applications
  Using defaults (no grid search needed)
  Default values are well-validated in literature


In [348]:
print("  Hyperparameters (literature-based defaults):")
print(f"\n  C: {ELASTICNET_C} (moderate regularization)")
print(f"  l1_ratio: {ELASTICNET_L1_RATIO} (70% L1 / 30% L2)")
print("\nThese values are standard in financial ML applications")
print("and provide a good balance between feature selection and stability.")


# Preprocessing: Standardize features
ct = ColumnTransformer([
    ("num", StandardScaler(with_mean=True), num_cols),
], remainder="drop", sparse_threshold=0.0)  # force dense for feature importance

# Classifier with ElasticNet regularization
clf = LogisticRegression(
    penalty='elasticnet',      # Use ElasticNet (L1 + L2)
    solver='saga',             # Only solver supporting elasticnet
    l1_ratio=ELASTICNET_L1_RATIO,  # Mix of L1 and L2
    C=ELASTICNET_C,            # Regularization strength
    max_iter=5000,             # More iterations for convergence
    tol=1e-4,
    random_state=42,
    n_jobs=-1                  # Use all CPUs
)

# Pipeline: Preprocessing → Classification
pipe = Pipeline([("prep", ct), ("clf", clf)])

print("\n  - Model configured with automatic feature selection")
print("  - Features with low importance will be set to zero during training")


  Hyperparameters (literature-based defaults):

  C: 0.1 (moderate regularization)
  l1_ratio: 0.7 (70% L1 / 30% L2)

These values are standard in financial ML applications
and provide a good balance between feature selection and stability.

  - Model configured with automatic feature selection
  - Features with low importance will be set to zero during training


# **Dynamic Feature Selection with ElasticNet**

In [349]:
pred_prob_up_new = pd.Series(index=df.index, dtype=float)
pred_prob_down_new = pd.Series(index=df.index, dtype=float) 
pred_score_new = pd.Series(index=df.index, dtype=float)
pred_class_new = pd.Series(index=df.index, dtype=int)
used_mask_new = pd.Series(False, index=df.index)

# Track feature selection across windows
feature_selection_history = []


print("Rolling Window Training With Feature Selection (In-Sample Only)")


window_num = 0
start_pos = 0

while start_pos + initial + horizon <= len(dates_insample):
    window_num += 1
    
    train_dates = dates_insample[start_pos: start_pos + initial]
    valid_dates = dates_insample[start_pos + initial: start_pos + initial + horizon]
    
    
    print(f"\nWindow {window_num}: {train_dates.min().date()} to {valid_dates.max().date()}")
    

    if fixedWindow:
        start_pos += step
    else:
        start_pos += step

    idx_tr = df.index.get_level_values("date").isin(train_dates)
    idx_va = df.index.get_level_values("date").isin(valid_dates)

    X_tr = df.loc[idx_tr, num_cols]
    y_tr = DIR_binary.loc[idx_tr]
    X_va = df.loc[idx_va, num_cols]
    y_va = DIR_binary.loc[idx_va]

    # Train model
    pipe.fit(X_tr, y_tr)
    
    # Feature Selection Analysis
    coef = pipe.named_steps["clf"].coef_[0]
    
    # Features with non-zero coefficients
    feature_importance = pd.DataFrame({
        'feature': num_cols,
        'coefficient': coef,
        'abs_coefficient': np.abs(coef)
    })
    
    # Count selected features (abs(coef) > threshold)
    ZERO_THRESHOLD = 1e-5
    selected_features = feature_importance[feature_importance['abs_coefficient'] > ZERO_THRESHOLD]
    n_selected = len(selected_features)
    pct_selected = (n_selected / len(num_cols)) * 100
    
    # Top 10 most important features
    top_features = selected_features.nlargest(10, 'abs_coefficient')
    
    # Store feature selection info
    feature_selection_history.append({
        'window': window_num,
        'n_features_selected': n_selected,
        'pct_selected': pct_selected,
        'selected_features': selected_features['feature'].tolist(),  # ALL selected features
        'top_5_features': top_features.head(5)['feature'].tolist(),
        'train_start': train_dates.min(),
        'valid_end': valid_dates.max()
    })
    
    # Predict
    proba = pipe.predict_proba(X_va)
    p_down = proba[:, 0]
    p_up = proba[:, 1]
    score = 2 * p_up - 1
    yhat = (p_up > 0.5).astype(int)
    
    # Store predictions
    pred_prob_up_new.loc[idx_va] = p_up
    pred_prob_down_new.loc[idx_va] = p_down
    pred_score_new.loc[idx_va] = score
    pred_class_new.loc[idx_va] = yhat
    used_mask_new.loc[idx_va] = True
    
    # Report
    accuracy = (yhat == y_va).mean()
    print(f"Training: {len(X_tr):,} samples")
    print(f"Validation: {len(X_va):,} samples, Accuracy: {accuracy:.1%}")
    print("\nFeature Selection:")
    print(f"   Selected: {n_selected}/{len(num_cols)} features ({pct_selected:.1f}%)")
    print("   Top 5 features by importance:")
    for i, row in top_features.head(5).iterrows():
        print(f"      {i+1}. {row['feature']}: {row['coefficient']:+.4f}")


print(f"✓ TRAINING COMPLETE: {window_num} windows processed")
print(f"✓ Total validated: {used_mask_new.sum():,} / {len(df):,}")


# Update global predictions with new ones
pred_prob_up = pred_prob_up_new
pred_prob_down = pred_prob_down_new
pred_score = pred_score_new
pred_class = pred_class_new
used_mask = used_mask_new


Rolling Window Training With Feature Selection (In-Sample Only)

Window 1: 2018-03-28 to 2019-12-31
Training: 26,307 samples
Validation: 8,769 samples, Accuracy: 51.9%

Feature Selection:
   Selected: 27/39 features (69.2%)
   Top 5 features by importance:
      20. hml: +0.1490
      18. mktrf: +0.0944
      19. smb: +0.0901
      29. ti_bb_percent_20_2: -0.0567
      37. ti_skew_63: -0.0559

Window 2: 2018-04-16 to 2020-01-17
Training: 26,307 samples
Validation: 8,769 samples, Accuracy: 50.7%

Feature Selection:
   Selected: 29/39 features (74.4%)
   Top 5 features by importance:
      20. hml: +0.1625
      18. mktrf: +0.1249
      19. smb: +0.0779
      36. ti_stoch_k_14_3_3: +0.0652
      39. ti_aroon_osc_25: +0.0560

Window 3: 2018-05-02 to 2020-02-05
Training: 26,307 samples
Validation: 8,769 samples, Accuracy: 51.7%

Feature Selection:
   Selected: 25/39 features (64.1%)
   Top 5 features by importance:
      20. hml: +0.1713
      18. mktrf: +0.1160
      22. umd: +0.0886
    

In [350]:
# Feature Selection Analysis
print("Feature Selection Summary Across All Windows")

# Create summary dataframe
fs_summary = pd.DataFrame(feature_selection_history)

print("\nOverall Statistics:")
print(f"  Average features selected: {fs_summary['n_features_selected'].mean():.1f} / {len(num_cols)}")
print(f"  Min features: {fs_summary['n_features_selected'].min()}")
print(f"  Max features: {fs_summary['n_features_selected'].max()}")
print(f"  Average selection rate: {fs_summary['pct_selected'].mean():.1f}%")

# Find features that appear most frequently in top 5
all_top_features = []
for features_list in fs_summary['top_5_features']:
    all_top_features.extend(features_list)

feature_frequency = pd.Series(all_top_features).value_counts()

print("\n  Most Consistently Important Features (appeared in top 5):")

for i, (feature, count) in enumerate(feature_frequency.head(15).items(), 1):
    pct = (count / len(fs_summary)) * 100
    bar = "█" * int(pct / 10)
    print(f"{i:2d}. {feature:<25} │ {count}/{len(fs_summary)} windows ({pct:5.1f}%) {bar}")


print("\nKey Insights:")


# Identify feature categories
technical_indicators = [f for f in feature_frequency.head(10).index if f.startswith('ti_')]
ff_factors = [f for f in feature_frequency.head(10).index if any(x in f for x in ['mktrf', 'smb', 'hml', 'umd', 'rf'])]
ibes_features = [f for f in feature_frequency.head(10).index if any(x in f for x in ['cons_', 'n_analysts', 'n_up', 'n_down'])]
lag_features = [f for f in feature_frequency.head(10).index if 'lag' in f.lower()]

print("\nTop 10 features by category:")
print(f"    Technical Indicators: {len(technical_indicators)}")
if technical_indicators:
    print(f"     → {', '.join(technical_indicators[:3])}")
print(f"    Fama-French Factors: {len(ff_factors)}")
if ff_factors:
    print(f"     → {', '.join(ff_factors[:3])}")
print(f"    IBES/Consensus: {len(ibes_features)}")
if ibes_features:
    print(f"     → {', '.join(ibes_features[:3])}")
print(f"     Lagged Features: {len(lag_features)}")
if lag_features:
    print(f"     → {', '.join(lag_features[:3])}")


Feature Selection Summary Across All Windows

Overall Statistics:
  Average features selected: 26.7 / 39
  Min features: 24
  Max features: 29
  Average selection rate: 68.5%

  Most Consistently Important Features (appeared in top 5):
 1. hml                       │ 10/10 windows (100.0%) ██████████
 2. ti_stoch_k_14_3_3         │ 8/10 windows ( 80.0%) ████████
 3. ti_cmf_20                 │ 5/10 windows ( 50.0%) █████
 4. mktrf                     │ 5/10 windows ( 50.0%) █████
 5. ti_rsi_14                 │ 4/10 windows ( 40.0%) ████
 6. smb                       │ 4/10 windows ( 40.0%) ████
 7. ti_rsi_28                 │ 4/10 windows ( 40.0%) ████
 8. umd                       │ 3/10 windows ( 30.0%) ███
 9. vol                       │ 3/10 windows ( 30.0%) ███
10. ti_skew_63                │ 2/10 windows ( 20.0%) ██
11. ti_bb_percent_20_2        │ 1/10 windows ( 10.0%) █
12. ti_aroon_osc_25           │ 1/10 windows ( 10.0%) █

Key Insights:

Top 10 features by category:
    Tech

In [351]:
# Feature Selection Fof Final Model
# After analyzing 10 rolling windows, select features for the final model
# trained on all in-sample data

print("\nFeature Selection For Final Model")

# Calculate Feature Frequencies Across All Windows

# Extract selection information from all windows
n_windows = len(feature_selection_history)
feature_selected_counts = {feat: 0 for feat in num_cols}

# Count how many windows each feature was selected in
for window_info in feature_selection_history:
    selected_features = window_info['selected_features']
    for feat in selected_features:
        feature_selected_counts[feat] += 1

# Calculate frequency (percentage of windows)
feature_freq = pd.DataFrame({
    'feature': list(feature_selected_counts.keys()),
    'count': list(feature_selected_counts.values()),
    'frequency': [count / n_windows for count in feature_selected_counts.values()]
}).sort_values('frequency', ascending=False)

# Selection Criteria: Frequency Threshold
# Keep features that appear in ≥X% of windows
# Recommended thresholds:
#   - 0.3 (30%) = Lenient (more features, captures regime-specific signals)
#   - 0.5 (50%) = Moderate (balanced, industry standard)
#   - 0.7 (70%) = Strict (fewer features, only stable predictors)

SELECTION_THRESHOLD = 0.5  # Moderate: 50% threshold

selected_features_mask = feature_freq['frequency'] >= SELECTION_THRESHOLD
final_feature_list = feature_freq.loc[selected_features_mask, 'feature'].tolist()

print("\n  Selection Criterion:")
print(f"   Frequency threshold: ≥ {SELECTION_THRESHOLD*100:.0f}% of windows")
print("   Rationale: Features must be consistently predictive across")
print("              different market regimes to be included")

print("\n  Results:")
print(f"   Features selected: {len(final_feature_list)} / {len(num_cols)}")
print(f"   Features removed:  {len(num_cols) - len(final_feature_list)}")
print(f"   Reduction: {(1 - len(final_feature_list)/len(num_cols))*100:.1f}%")


# Display Selected Features

print(f"\n  Selected Features ({len(final_feature_list)}):")
print(f"{'Feature':<30} {'Frequency':<12} {'Appearances':<15}")


selected_features_df = feature_freq[selected_features_mask].copy()
for idx, row in selected_features_df.iterrows():
    feat = row['feature']
    freq = row['frequency']
    count = row['count']
    bar = '█' * int(freq * 10)
    print(f"{feat:<30} {freq:>6.1%} ({count:>2}/{n_windows})   {bar}")


# Display Removed Features

removed_features_df = feature_freq[~selected_features_mask].copy()

if len(removed_features_df) > 0:
    print(f"\n  REMOVED FEATURES ({len(removed_features_df)}):")
    print(f"{'Feature':<30} {'Frequency':<12} {'Appearances':<15}")
    
    for idx, row in removed_features_df.iterrows():
        feat = row['feature']
        freq = row['frequency']
        count = row['count']
        bar = '░' * int(freq * 10)
        print(f"{feat:<30} {freq:>6.1%} ({count:>2}/{n_windows})   {bar}")

# Feature Categories Breakdown

def categorize_feature(feat):
    """Categorize feature by type"""
    if feat.startswith('ti_'):
        return 'Technical Indicators'
    elif feat in ['mktrf', 'smb', 'hml', 'rf', 'umd']:
        return 'Fama-French Factors'
    elif feat.startswith('cons_') or feat.startswith('n_'):
        return 'IBES Consensus'
    elif feat.startswith('adjclose_lag'):
        return 'Price Lags'
    elif feat in ['adj_mktcap', 'vol', 'retx']:
        return 'Market Data'
    else:
        return 'Other'

selected_features_df['category'] = selected_features_df['feature'].apply(categorize_feature)
category_counts = selected_features_df['category'].value_counts()

print("\n  Selected Features By Category:")

for category, count in category_counts.items():
    pct = count / len(selected_features_df) * 100
    print(f"  {category:<30} {count:>2} features ({pct:>5.1f}%)")


# Academic Justification

print("\n  Justification For Report:")

print("\"Features were selected using a frequency-based criterion: only those")
print(f"appearing in ≥{SELECTION_THRESHOLD*100:.0f}% of cross-validation windows were retained.")
print("This ensures the final model uses consistently predictive features")
print("that are robust across different market regimes. This approach follows")
print("Meinshausen & Bühlmann (2010) stability selection methodology.\"")

print("\n  Reference:")
print("   Meinshausen, N., & Bühlmann, P. (2010). Stability selection.")
print("   Journal of the Royal Statistical Society: Series B, 72(4), 417-473.")


# Save For Use In Final Model Training (Cell 17)

print("\n✓ Feature list saved to 'final_feature_list' variable")
print("  This will be used to train the final model on all in-sample data")



Feature Selection For Final Model

  Selection Criterion:
   Frequency threshold: ≥ 50% of windows
   Rationale: Features must be consistently predictive across
              different market regimes to be included

  Results:
   Features selected: 27 / 39
   Features removed:  12
   Reduction: 30.8%

  Selected Features (27):
Feature                        Frequency    Appearances    
vol                            100.0% (10/10)   ██████████
smb                            100.0% (10/10)   ██████████
hml                            100.0% (10/10)   ██████████
umd                            100.0% (10/10)   ██████████
mktrf                          100.0% (10/10)   ██████████
adjclose_lag0                  100.0% (10/10)   ██████████
cons_high                      100.0% (10/10)   ██████████
ti_kurtosis_63                 100.0% (10/10)   ██████████
ti_skew_63                     100.0% (10/10)   ██████████
ti_aroon_osc_25                100.0% (10/10)   ██████████
ti_variance_21      

In [352]:
# Suppress numpy warnings for empty slices (occurs when calculating volatility for dates with no data)
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')


In [353]:
# STRATEGY 2: RANKING-BASED PORTFOLIO (RECOMMENDED)
# Rank stocks by score and go long top X%, short bottom X%

def build_ranking_portfolio(group_scores, long_pct=0.20, short_pct=0.20):
    """
    Build portfolio by ranking stocks within each date.
    
    Parameters:
    -----------
    group_scores : pd.Series
        Scores for one date (all stocks)
    long_pct : float
        Percentage of stocks to go long (e.g., 0.20 = top 20%)
    short_pct : float
        Percentage of stocks to go short (e.g., 0.20 = bottom 20%)
    
    Returns:
    --------
    pd.Series : Raw weights (equal-weighted within long/short buckets)
    """
    n_stocks = len(group_scores)
    n_long = max(1, int(n_stocks * long_pct))
    n_short = max(1, int(n_stocks * short_pct))
    
    # Rank stocks by score
    sorted_idx = group_scores.sort_values(ascending=False).index
    
    # Assign equal weights within buckets
    weights = pd.Series(0.0, index=group_scores.index)
    weights.loc[sorted_idx[:n_long]] = 1.0 / n_long      # Top N% = Long
    weights.loc[sorted_idx[-n_short:]] = -1.0 / n_short  # Bottom N% = Short
    
    return weights

# Apply ranking strategy to each date
valid_df = df.loc[used_mask].copy()
valid_df['score'] = pred_score.loc[used_mask]

# Group by date and apply ranking strategy
# Note: We use transform() instead of apply() to avoid index duplication issues
def apply_ranking_by_date(group):
    """Apply ranking strategy to a group (one date)"""
    scores = group['score']
    return build_ranking_portfolio(scores, long_pct=0.20, short_pct=0.20)

# Apply ranking to each date
weights_ranking = valid_df.groupby(level='date', group_keys=False).apply(
    apply_ranking_by_date
)

# Flatten if needed (groupby.apply can create nested structure)
if isinstance(weights_ranking, pd.DataFrame):
    weights_ranking = weights_ranking.iloc[:, 0]

print("\nSTRATEGY 2: RANKING-BASED (TOP/BOTTOM 20%)")
print(f"  Total positions (before normalization): {(weights_ranking != 0).sum():,}")
print(f"  Long positions: {(weights_ranking > 0).sum():,}")
print(f"  Short positions: {(weights_ranking < 0).sum():,}")

# Show example for one date (use level number to avoid ambiguity)
example_date = valid_df.index.get_level_values(1).unique()[0]  # level 1 = 'date'
example_idx = valid_df.index.get_level_values(1) == example_date
example_weights = weights_ranking.loc[example_idx]
print(f"\nExample for {example_date.date()}:")
print(f"  Number of longs: {(example_weights > 0).sum()}")
print(f"  Number of shorts: {(example_weights < 0).sum()}")
print(f"  Sum of longs: {example_weights[example_weights > 0].sum():.4f}")
print(f"  Sum of shorts: {example_weights[example_weights < 0].sum():.4f}")
print(f"  Net exposure (should be ~0): {example_weights.sum():.4f}")



STRATEGY 2: RANKING-BASED (TOP/BOTTOM 20%)
  Total positions (before normalization): 6,570
  Long positions: 3,285
  Short positions: 3,285

Example for 2019-07-25:
  Number of longs: 13
  Number of shorts: 14
  Sum of longs: 0.8667
  Sum of shorts: -0.9333
  Net exposure (should be ~0): -0.0667


In [354]:
# Volatility-Adjusted Position Sizing
# Goal: Give smaller positions to high-volatility stocks, larger to low-vol stocks
# This ensures equal risk contribution across positions

print("Step 1: Calculate Rolling Volatility")

# Calculate rolling volatility for each stock
# Use 20-day rolling window (approx 1 month)
VOL_WINDOW = 20

def calculate_rolling_volatility(group):
    """Calculate rolling std of returns for each stock"""
    # Use adjclose_lead (log returns) to calculate volatility
    rolling_vol = group['adjclose_lead'].rolling(window=VOL_WINDOW, min_periods=VOL_WINDOW//2).std()
    # Annualize: std(daily) * sqrt(252)
    return rolling_vol * np.sqrt(252)

# Calculate volatility per stock (permno)
valid_df['volatility'] = valid_df.groupby(level='permno', group_keys=False).apply(
    calculate_rolling_volatility
)

# Fill any NaN volatilities with cross-sectional median
valid_df['volatility'] = valid_df.groupby(level='date')['volatility'].transform(
    lambda x: x.fillna(x.median())
)

print("Volatility statistics:")
print(f"  Mean: {valid_df['volatility'].mean():.2%}")
print(f"  Median: {valid_df['volatility'].median():.2%}")
print(f"  Min: {valid_df['volatility'].min():.2%}")
print(f"  Max: {valid_df['volatility'].max():.2%}")
print(f"  25th percentile: {valid_df['volatility'].quantile(0.25):.2%}")
print(f"  75th percentile: {valid_df['volatility'].quantile(0.75):.2%}")


print("\nStep 2: Apply Volatility Adjustment")


# Store raw ranking weights
valid_df['weight_raw'] = weights_ranking

# Apply inverse volatility weighting
# Inverse vol: w_adjusted = w_raw / volatility
# This gives larger weights to low-vol stocks, smaller to high-vol stocks
valid_df['weight_vol_adjusted'] = valid_df['weight_raw'] / valid_df['volatility']

print("Before vol adjustment:")
print(f"  Weight std: {valid_df['weight_raw'].std():.6f}")
print("\nAfter vol adjustment (before normalization):")
print(f"  Weight std: {valid_df['weight_vol_adjusted'].std():.6f}")


print("\nStep 3: Normalize To Dollar Neutrality")


def normalize_weights_to_dollar_neutral(group):
    """
    Normalize weights per date to achieve dollar neutrality.
    Long side = 0.5, Short side = -0.5
    """
    long_mask = group > 0
    short_mask = group < 0
    
    long_sum = group[long_mask].sum()
    short_sum = abs(group[short_mask].sum())
    
    normalized = group.copy()
    
    # Scale longs to sum to 0.5
    if long_sum > 0:
        normalized[long_mask] = group[long_mask] / long_sum * 0.5
    
    # Scale shorts to sum to -0.5
    if short_sum > 0:
        normalized[short_mask] = group[short_mask] / short_sum * (-0.5)
    
    return normalized

# Apply normalization per date
valid_df['weight'] = valid_df.groupby(level=1)['weight_vol_adjusted'].transform(
    normalize_weights_to_dollar_neutral
)

print("After normalization:")
print(f"  Total weight sum: {valid_df['weight'].sum():.6f} (target: 0.00)")
print(f"  Max single position: {valid_df['weight'].abs().max():.4f}")
print(f"  Min single position (non-zero): {valid_df[valid_df['weight'] != 0]['weight'].abs().min():.4f}")

# Compare equal-weight vs vol-adjusted
check_date_idx = 10
check_date = valid_df.index.get_level_values(1).unique()[check_date_idx]
date_mask = valid_df.index.get_level_values(1) == check_date


print(f"\nComparison ForR {check_date.date()}")


# Get top 3 longs by raw weight (equal-weighted)
raw_weights = valid_df.loc[date_mask, ['weight_raw', 'weight', 'volatility', 'ticker']]
raw_weights = raw_weights[raw_weights['weight_raw'] > 0].sort_values('weight_raw', ascending=False).head(3)

print("Top 3 Long Positions (equal-weight vs vol-adjusted):")
print(f"{'Ticker':<8} {'Vol':<8} {'Raw Weight':<12} {'Vol-Adj Weight':<15} {'Change'}")

for idx, row in raw_weights.iterrows():
    ticker = row['ticker']
    vol = row['volatility']
    raw_w = row['weight_raw']
    adj_w = row['weight']
    change = ((adj_w / raw_w) - 1) * 100 if raw_w != 0 else 0
    print(f"\n{ticker:<8} {vol:>6.1%} {raw_w:>11.4f} {adj_w:>14.4f} {change:>+7.1f}%")

# Verify dollar neutrality
date_weights = valid_df.loc[date_mask, 'weight']
print("\nPortfolio check:")
print(f"  Long exposure: {date_weights[date_weights > 0].sum():.4f} (target: 0.50)")
print(f"  Short exposure: {date_weights[date_weights < 0].sum():.4f} (target: -0.50)")
print(f"  Net exposure: {date_weights.sum():.6f} (target: 0.00)")
print(f"  Gross exposure: {date_weights.abs().sum():.4f} (target: 1.00)")

Step 1: Calculate Rolling Volatility
Volatility statistics:
  Mean: 37.45%
  Median: 26.23%
  Min: 5.36%
  Max: 427.82%
  25th percentile: 17.66%
  75th percentile: 43.42%

Step 2: Apply Volatility Adjustment
Before vol adjustment:
  Weight std: 0.041084

After vol adjustment (before normalization):
  Weight std: 0.199589

Step 3: Normalize To Dollar Neutrality
After normalization:
  Total weight sum: 210.000000 (target: 0.00)
  Max single position: 0.0875
  Min single position (non-zero): 0.0043

Comparison ForR 2019-08-08
Top 3 Long Positions (equal-weight vs vol-adjusted):
Ticker   Vol      Raw Weight   Vol-Adj Weight  Change

HON       24.7%      0.0667         0.0373   -44.0%

ETN       31.1%      0.0667         0.0296   -55.7%

PANW      22.2%      0.0667         0.0414   -37.8%

Portfolio check:
  Long exposure: 1.0000 (target: 0.50)
  Short exposure: 0.0000 (target: -0.50)
  Net exposure: 1.000000 (target: 0.00)
  Gross exposure: 1.0000 (target: 1.00)


In [355]:
# Calculate Portfolio Returns
# Portfolio return = sum of (normalized_weight * stock_return) across all stocks each day

# Per-date portfolio log-return (using normalized weights!)
port_ret_by_date = (valid_df["weight"] * valid_df["adjclose_lead"]).groupby(level="date").sum()

# Build equity curve
equity = np.exp(port_ret_by_date.cumsum())
equity.index.name = "date"


print("\nPortfolio Returns Calculated")
print(f"\nTrading days: {len(port_ret_by_date)}")
print(f"  Date range: {port_ret_by_date.index.min().date()} to {port_ret_by_date.index.max().date()}")
print("\nDaily return statistics:")
print(f"  Mean: {port_ret_by_date.mean():.6f}")
print(f"  Std: {port_ret_by_date.std():.6f}")
print(f"  Min: {port_ret_by_date.min():.6f}")
print(f"  Max: {port_ret_by_date.max():.6f}")
print("\nEquity curve:")
print(f"  Initial: {equity.iloc[0]:.4f}")
print(f"  Final: {equity.iloc[-1]:.4f}")
print(f"  Total return: {(equity.iloc[-1] - 1) * 100:.2f}%")


Portfolio Returns Calculated

Trading days: 219
  Date range: 2019-07-25 to 2020-06-05

Daily return statistics:
  Mean: 0.000194
  Std: 0.022520
  Min: -0.126934
  Max: 0.106110

Equity curve:
  Initial: 1.0000
  Final: 1.0433
  Total return: 4.33%


In [356]:
# Long-Only Strategy (for comparison)
# Create a long-only version: only buy the top-ranked stocks

print("Building Long-Only Strategy")

# Parameter: what % of stocks to hold long
LONG_ONLY_PCT = 0.30  # Hold top 30% of stocks

def build_long_only_portfolio(group_scores, long_pct=0.30):
    """
    Build long-only portfolio by ranking stocks.
    Only buys the top X% of stocks by score.
    """
    n_stocks = len(group_scores)
    n_long = max(1, int(n_stocks * long_pct))
    
    # Rank stocks by score
    sorted_idx = group_scores.sort_values(ascending=False).index
    
    # Assign equal weights to top stocks
    weights = pd.Series(0.0, index=group_scores.index)
    weights.loc[sorted_idx[:n_long]] = 1.0 / n_long  # Equal weight, sums to 1.0
    
    return weights

# Apply long-only strategy
def apply_long_only_by_date(group):
    """Apply long-only ranking to a group (one date)"""
    scores = group['score']
    return build_long_only_portfolio(scores, long_pct=LONG_ONLY_PCT)

weights_long_only = valid_df.groupby(level=1, group_keys=False).apply(
    apply_long_only_by_date
)

# Flatten if needed
if isinstance(weights_long_only, pd.DataFrame):
    weights_long_only = weights_long_only.iloc[:, 0]

valid_df['weight_long_only_raw'] = weights_long_only

print(f"Long-only strategy: Top {LONG_ONLY_PCT*100:.0f}% of stocks")
print(f"Total positions: {(weights_long_only != 0).sum():,}")
print(f"Average positions per date: {(weights_long_only != 0).sum() / len(valid_df.index.get_level_values(1).unique()):.1f}")

# Apply volatility adjustment to long-only weights
valid_df['weight_long_only_vol'] = valid_df['weight_long_only_raw'] / valid_df['volatility']

# Normalize so weights sum to 1.0 (100% invested)
def normalize_long_only(group):
    """Normalize weights to sum to 1.0 (fully invested)"""
    total = group.sum()
    if total > 0:
        return group / total
    return group

valid_df['weight_long_only'] = valid_df.groupby(level=1)['weight_long_only_vol'].transform(
    normalize_long_only
)

print("\nAfter normalization:")
print(f"  Sum of weights (should be ~1.0 per date): {valid_df.groupby(level=1)['weight_long_only'].sum().mean():.4f}")
print(f"  Max position: {valid_df['weight_long_only'].max():.2%}")
print(f"  Min position (non-zero): {valid_df[valid_df['weight_long_only'] > 0]['weight_long_only'].min():.2%}")

# Calculate long-only returns
port_ret_long_only = (valid_df["weight_long_only"] * valid_df["adjclose_lead"]).groupby(level=1).sum()
equity_long_only = np.exp(port_ret_long_only.cumsum())
equity_long_only.index.name = "date"

print("\nLong-Only Performance:")
print(f"  Trading days: {len(port_ret_long_only)}")
print(f"  Initial equity: {equity_long_only.iloc[0]:.4f}")
print(f"  Final equity: {equity_long_only.iloc[-1]:.4f}")
print(f"  Total return: {(equity_long_only.iloc[-1] - 1) * 100:.2f}%")


Building Long-Only Strategy
Long-only strategy: Top 30% of stocks
Total positions: 5,037
Average positions per date: 23.0

After normalization:
  Sum of weights (should be ~1.0 per date): 0.9589
  Max position: 11.96%
  Min position (non-zero): 0.49%

Long-Only Performance:
  Trading days: 219
  Initial equity: 1.0000
  Final equity: 0.9742
  Total return: -2.58%


In [357]:
# Step 1: Train Final Model On All In-Sample Data
print("\nTraining Final Model On All In-Sample Data")

# Rebuild Pipeline With Selected Features Only

# Create new pipeline using only the selected features from Cell 9
ct_final = ColumnTransformer([
    ("num", StandardScaler(with_mean=False), final_feature_list)  # Use selected features
], remainder="drop")

clf_final = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    l1_ratio=ELASTICNET_L1_RATIO,
    C=ELASTICNET_C,
    max_iter=5000,
    tol=1e-4,
    random_state=42,
    n_jobs=-1
)

pipe_final = Pipeline([("prep", ct_final), ("clf", clf_final)])

print(f"\n✓ Pipeline rebuilt with {len(final_feature_list)} selected features")
print(f"  (Original pipeline used all {len(num_cols)} features)")

# Train On All In-Sample Data

# Get all in-sample data
insample_mask = df.index.get_level_values('date').isin(dates_insample)
X_train_final = df.loc[insample_mask, final_feature_list]
y_train_final = DIR_binary.loc[insample_mask]

# Train final model
pipe_final.fit(X_train_final, y_train_final)


# Analyze Final Model

# Check which of the selected features ElasticNet actually uses
final_coef = pipe_final.named_steps["clf"].coef_[0]
non_zero_mask = np.abs(final_coef) > 1e-5
actually_used_features = np.array(final_feature_list)[non_zero_mask]

print("\nFinal model trained on:")
print(f"  Period: {dates_insample.min().date()} to {dates_insample.max().date()}")
print(f"  Days: {len(dates_insample)}")
print(f"  Samples: {len(X_train_final):,}")

print("\nFeature Selection (2-stage):")
print(f"  Stage 1 (Frequency-based): {len(final_feature_list)} / {len(num_cols)} features selected")
print(f"  Stage 2 (ElasticNet on final model): {len(actually_used_features)} / {len(final_feature_list)} features used")
print(f"  Total reduction: {len(num_cols)} → {len(actually_used_features)} ({len(actually_used_features)/len(num_cols)*100:.1f}%)")

print("\nTop 10 features by coefficient magnitude:")
feature_importance = pd.DataFrame({
    'feature': final_feature_list,
    'coefficient': final_coef,
    'abs_coefficient': np.abs(final_coef)
}).sort_values('abs_coefficient', ascending=False)

for i, (idx, row) in enumerate(feature_importance.head(10).iterrows(), 1):
    feat = row['feature']
    coef = row['coefficient']
    print(f"  {i:2d}. {feat:25s} {coef:+.4f}")



Training Final Model On All In-Sample Data

✓ Pipeline rebuilt with 27 selected features
  (Original pipeline used all 39 features)

Final model trained on:
  Period: 2018-03-28 to 2020-06-11
  Days: 556
  Samples: 43,924

Feature Selection (2-stage):
  Stage 1 (Frequency-based): 27 / 39 features selected
  Stage 2 (ElasticNet on final model): 27 / 27 features used
  Total reduction: 39 → 27 (69.2%)

Top 10 features by coefficient magnitude:
   1. hml                       +0.1412
   2. adjclose_lag0             -0.1263
   3. umd                       +0.1080
   4. mktrf                     +0.0985
   5. ti_rsi_14                 +0.0751
   6. adjclose_lag3             -0.0680
   7. ti_stoch_k_14_3_3         +0.0560
   8. ti_rsi_28                 -0.0556
   9. vol                       -0.0421
  10. ti_cmf_20                 -0.0342


In [358]:
# STEP 2: OUT-OF-SAMPLE EVALUATION (HELD-OUT TEST SET)

print("\nOut-Of-Sample Evaluation (Final Test)")
print("   WARNING: This data has NEVER been seen during training!")

# Get out-of-sample data
outofsample_mask = df.index.get_level_values('date').isin(dates_outofsample)
X_test = df.loc[outofsample_mask, final_feature_list]
y_test = DIR_binary.loc[outofsample_mask]

print("\nOut-of-sample period:")
print(f"  Dates: {dates_outofsample.min().date()} to {dates_outofsample.max().date()}")
print(f"  Days: {len(dates_outofsample)}")
print(f"  Samples: {len(X_test):,}")

# Predict using the FINAL model
test_prob_up = pipe_final.predict_proba(X_test)[:, 1]
test_prob_down = 1 - test_prob_up
test_score = 2 * test_prob_up - 1
test_pred = pipe_final.predict(X_test)

# Classification accuracy
test_accuracy = (test_pred == y_test).mean()
print("\nClassification Performance:")
print(f"  Accuracy: {test_accuracy:.1%}")
print(f"  Up class accuracy: {(test_pred[y_test == 1] == 1).mean():.1%}")
print(f"  Down class accuracy: {(test_pred[y_test == 0] == 0).mean():.1%}")

# Create test predictions series
pred_prob_up_test = pd.Series(test_prob_up, index=X_test.index, name='prob_up')
pred_score_test = pd.Series(test_score, index=X_test.index, name='score')
pred_class_test = pd.Series(test_pred, index=X_test.index, name='pred_class')

print(f"\nTest predictions created for {len(pred_score_test):,} observations")



Out-Of-Sample Evaluation (Final Test)

Out-of-sample period:
  Dates: 2020-06-12 to 2020-12-30
  Days: 140
  Samples: 11,060

Classification Performance:
  Accuracy: 53.6%
  Up class accuracy: 83.4%
  Down class accuracy: 18.7%

Test predictions created for 11,060 observations


In [359]:
# STEP 3: BUILD PORTFOLIO ON OUT-OF-SAMPLE PREDICTIONS

print("\nBuilding Portfolio On Out-Of-Sample Data")

# Get test data with predictions
test_df = df.loc[outofsample_mask].copy()
test_df['score'] = pred_score_test
test_df['pred_class'] = pred_class_test
test_df['prob_up'] = pred_prob_up_test

# Apply ranking-based portfolio construction (same as in-sample)
def build_ranking_portfolio_oos(group_scores, long_pct=0.20, short_pct=0.20):
    n_stocks = len(group_scores)
    n_long = max(1, int(n_stocks * long_pct))
    n_short = max(1, int(n_stocks * short_pct))
    
    sorted_idx = group_scores.sort_values(ascending=False).index
    
    weights = pd.Series(0.0, index=group_scores.index)
    weights.loc[sorted_idx[:n_long]] = 1.0 / n_long
    weights.loc[sorted_idx[-n_short:]] = -1.0 / n_short
    
    return weights

# Apply ranking to each date
weights_oos_raw = test_df.groupby(level='date', group_keys=False).apply(
    lambda g: build_ranking_portfolio_oos(g['score'], long_pct=0.20, short_pct=0.20)
)

test_df['weight_raw'] = weights_oos_raw

# Calculate volatility
test_df['volatility'] = test_df.groupby(level='permno')['adjclose_lag0'].transform(
    lambda x: x.rolling(window=20, min_periods=1).std() * np.sqrt(252)
)

# Fill NaN volatilities
test_df['volatility'] = test_df.groupby(level='date')['volatility'].transform(
    lambda x: x.fillna(x.median())
)
overall_median_vol = test_df['volatility'].median()
test_df['volatility'] = test_df['volatility'].fillna(overall_median_vol)

# Apply inverse volatility weighting
test_df['weight_vol_adj'] = test_df['weight_raw'] / test_df['volatility']

# Normalize to dollar neutrality
def normalize_weights_oos(group_df):
    long_mask = group_df['weight_vol_adj'] > 0
    short_mask = group_df['weight_vol_adj'] < 0
    
    long_sum = group_df.loc[long_mask, 'weight_vol_adj'].sum()
    short_sum = group_df.loc[short_mask, 'weight_vol_adj'].sum()
    
    normalized = group_df['weight_vol_adj'].copy()
    
    if long_sum > 0:
        normalized.loc[long_mask] = group_df.loc[long_mask, 'weight_vol_adj'] / long_sum * 0.5
    if short_sum < 0:
        normalized.loc[short_mask] = group_df.loc[short_mask, 'weight_vol_adj'] / abs(short_sum) * (-0.5)
    
    return normalized

test_df['weight'] = test_df.groupby(level='date', group_keys=False).apply(
    normalize_weights_oos
).values

# Calculate simple returns
test_df['simple_ret'] = np.exp(test_df['adjclose_lead']) - 1.0

# Calculate portfolio returns
port_ret_oos = (test_df['weight'] * test_df['simple_ret']).groupby(level='date').sum()
equity_oos = (1 + port_ret_oos).cumprod()

# Calculate metrics
sharpe_oos = port_ret_oos.mean() / port_ret_oos.std() * np.sqrt(252)
total_ret_oos = (equity_oos.iloc[-1] - 1) * 100
max_dd_oos = (equity_oos / equity_oos.cummax() - 1).min() * 100

print("\nOut-of-Sample Portfolio Statistics:")
print(f"  Trading days: {len(port_ret_oos)}")
print(f"  Avg positions per day: {(test_df['weight'] != 0).groupby(level='date').sum().mean():.1f}")
print(f"  Avg long positions: {(test_df['weight'] > 0).groupby(level='date').sum().mean():.1f}")
print(f"  Avg short positions: {(test_df['weight'] < 0).groupby(level='date').sum().mean():.1f}")

print("\nOut-of-Sample Performance:")
print(f"  Total Return:  {total_ret_oos:>7.2f}%")
print(f"  Sharpe Ratio:  {sharpe_oos:>7.2f}")
print(f"  Max Drawdown:  {max_dd_oos:>7.2f}%")
print(f"  Avg Daily Ret: {port_ret_oos.mean()*100:>7.4f}%")
print(f"  Daily Volatility: {port_ret_oos.std()*100:>7.4f}%")



Building Portfolio On Out-Of-Sample Data

Out-of-Sample Portfolio Statistics:
  Trading days: 140
  Avg positions per day: 30.0
  Avg long positions: 30.0
  Avg short positions: 0.0

Out-of-Sample Performance:
  Total Return:    22.91%
  Sharpe Ratio:     1.99
  Max Drawdown:   -13.12%
  Avg Daily Ret:  0.1552%
  Daily Volatility:  1.2405%


In [360]:
# FINAL COMPARISON: IN-SAMPLE VS OUT-OF-SAMPLE
print("\nPERFORMANCE COMPARISON: IN-SAMPLE VS OUT-OF-SAMPLE")

# In-sample metrics (from rolling CV on validation windows)
sharpe_insample = port_ret_by_date.mean() / port_ret_by_date.std() * np.sqrt(252)
ret_insample = (equity.iloc[-1] - 1) * 100
dd_insample = (equity / equity.cummax() - 1).min() * 100

# Create comparison table
comparison_data = {
    'In-Sample (Dev)': [
        f"{dates_insample.min().date()} to {dates_insample.max().date()}",
        f"{len(dates_insample)} days",
        f"{ret_insample:.2f}%",
        f"{sharpe_insample:.2f}",
        f"{dd_insample:.2f}%",
        f"{port_ret_by_date.mean()*100:.4f}%",
        f"{port_ret_by_date.std()*100:.4f}%"
    ],
    'Out-of-Sample (Test)': [
        f"{dates_outofsample.min().date()} to {dates_outofsample.max().date()}",
        f"{len(dates_outofsample)} days",
        f"{total_ret_oos:.2f}%",
        f"{sharpe_oos:.2f}",
        f"{max_dd_oos:.2f}%",
        f"{port_ret_oos.mean()*100:.4f}%",
        f"{port_ret_oos.std()*100:.4f}%"
    ]
}

comparison_df = pd.DataFrame(
    comparison_data,
    index=['Period', 'Duration', 'Total Return', 'Sharpe Ratio', 'Max Drawdown', 'Avg Daily Return', 'Daily Volatility']
)

print("\n" + comparison_df.to_string())

# Calculate degradation
sharpe_degradation = ((sharpe_oos - sharpe_insample) / sharpe_insample * 100) if sharpe_insample != 0 else 0
ret_degradation = ((total_ret_oos - ret_insample) / ret_insample * 100) if ret_insample != 0 else 0

print("\nPerformance Degradation Analysis:")
print(f"  Sharpe Ratio:  {sharpe_degradation:+.1f}% change")
print(f"  Total Return:  {ret_degradation:+.1f}% change")

if abs(sharpe_degradation) < 20:
    assessment = "Excellent - Minimal degradation, model generalizes well"
elif abs(sharpe_degradation) < 40:
    assessment = "Good - Acceptable degradation for financial ML"
elif abs(sharpe_degradation) < 60:
    assessment = "Moderate - Some overfitting, but within normal range"
else:
    assessment = "Concerning - Significant overfitting detected"

print(f"\n  Assessment: {assessment}")

print("\n   Critical Interpretation:")
print("  - IN-SAMPLE: Used for model development (may be biased)")
print("  - OUT-OF-SAMPLE: TRUE expected performance (unbiased)")
print("  - Report OUT-OF-SAMPLE as your primary result!")
print("  - In-sample helps understand overfitting, not true performance")



PERFORMANCE COMPARISON: IN-SAMPLE VS OUT-OF-SAMPLE

                           In-Sample (Dev)      Out-of-Sample (Test)
Period            2018-03-28 to 2020-06-11  2020-06-12 to 2020-12-30
Duration                          556 days                  140 days
Total Return                         4.33%                    22.91%
Sharpe Ratio                          0.14                      1.99
Max Drawdown                       -34.60%                   -13.12%
Avg Daily Return                   0.0194%                   0.1552%
Daily Volatility                   2.2520%                   1.2405%

Performance Degradation Analysis:
  Sharpe Ratio:  +1355.4% change
  Total Return:  +429.0% change

  Assessment: Concerning - Significant overfitting detected

   Critical Interpretation:
  - IN-SAMPLE: Used for model development (may be biased)
  - OUT-OF-SAMPLE: TRUE expected performance (unbiased)
  - Report OUT-OF-SAMPLE as your primary result!
  - In-sample helps understand overfitting

In [361]:
# Generate HTML Reports: Out-Of-Sample Only (Both Strategies)

import quantstats as qs
from src.helpers._extract import ensure_dir


print("\nGenerating Out-Of-Sample HTML Reports")
print("   These reports show TRUE out-of-sample performance only!")

ensure_dir("out")

# Prepare Data: Extract Fama-French factors from out-of-sample period

used_mask_oos = test_df['weight'] != 0

rf_oos = (
    test_df.loc[used_mask_oos]
    .reset_index()[["date", "rf"]]
    .dropna()
    .groupby("date", as_index=True)["rf"]
    .mean()
    .astype(float)
    .sort_index()
)

mktrf_oos = (
    test_df.loc[used_mask_oos]
    .reset_index()[["date", "mktrf"]]
    .dropna()
    .groupby("date", as_index=True)["mktrf"]
    .mean()
    .astype(float)
    .sort_index()
)


# Report 1: OUT-OF-SAMPLE LONG-SHORT (Market Neutral)

print("\n1. Generating Out-Of-Sample Long-Short report...")

# Convert equity to returns
strategy_rets_ls = equity_oos.pct_change().dropna()
strategy_rets_ls.name = "Long-Short Market Neutral"

# Prepare risk-free and benchmark
rf_series_oos = rf_oos.reindex(strategy_rets_ls.index).ffill().bfill()
bench_rets_oos = (mktrf_oos + rf_oos).reindex(strategy_rets_ls.index).ffill().bfill()

# Convert to excess returns
strategy_excess_ls = (strategy_rets_ls - rf_series_oos).dropna()
bench_excess_oos = (bench_rets_oos - rf_series_oos).reindex(strategy_excess_ls.index).dropna()

# Align dates
common_idx_ls = strategy_excess_ls.index.intersection(bench_excess_oos.index)
strategy_excess_ls = strategy_excess_ls.reindex(common_idx_ls)
bench_excess_ls = bench_excess_oos.reindex(common_idx_ls)

qs.reports.html(
    strategy_excess_ls,
    benchmark=bench_excess_ls.to_frame("Market"),
    rf=0.0,
    periods_per_year=252,
    output="out/oos_long_short_tearsheet.html",
    title=f"OUT-OF-SAMPLE: Long-Short Market Neutral ({dates_outofsample.min().date()} to {dates_outofsample.max().date()})"
)
print("   ✓ Saved: out/oos_long_short_tearsheet.html")
print("   Strategy: Top 20% Long, Bottom 20% Short (Market Neutral)")
print(f"   Period: {strategy_excess_ls.index.min().date()} to {strategy_excess_ls.index.max().date()}")
print(f"   Days: {len(strategy_excess_ls)}")


# REPORT 2: OUT-OF-SAMPLE LONG-ONLY

print("\n2. Generating Out-Of-Sample Long-Only report...")

# Build long-only portfolio for out-of-sample period
def build_long_only_portfolio_oos(group_scores, long_pct=0.30):
    n_stocks = len(group_scores)
    n_long = max(1, int(n_stocks * long_pct))
    sorted_idx = group_scores.sort_values(ascending=False).index
    weights = pd.Series(0.0, index=group_scores.index)
    weights.loc[sorted_idx[:n_long]] = 1.0 / n_long
    return weights

# Apply long-only strategy to out-of-sample
weights_lo_oos_raw = test_df.groupby(level='date', group_keys=False).apply(
    lambda g: build_long_only_portfolio_oos(g['score'], long_pct=0.30)
)

test_df['weight_lo_raw'] = weights_lo_oos_raw

# Apply volatility adjustment
test_df['weight_lo_vol'] = test_df['weight_lo_raw'] / test_df['volatility']

# Normalize to sum to 1.0 (fully invested)
def normalize_long_only_oos(group):
    total = group.sum()
    if total > 0:
        return group / total
    return group

test_df['weight_lo'] = test_df.groupby(level='date')['weight_lo_vol'].transform(
    normalize_long_only_oos
)

# Calculate long-only returns
port_ret_lo_oos = (test_df['weight_lo'] * test_df['simple_ret']).groupby(level='date').sum()
equity_lo_oos = (1 + port_ret_lo_oos).cumprod()

# Convert to returns
strategy_rets_lo = equity_lo_oos.pct_change().dropna()
strategy_rets_lo.name = "Long-Only"

# Prepare risk-free and benchmark
rf_series_lo = rf_oos.reindex(strategy_rets_lo.index).ffill().bfill()
bench_rets_lo = (mktrf_oos + rf_oos).reindex(strategy_rets_lo.index).ffill().bfill()

# Convert to excess returns
strategy_excess_lo = (strategy_rets_lo - rf_series_lo).dropna()
bench_excess_lo = (bench_rets_lo - rf_series_lo).reindex(strategy_excess_lo.index).dropna()

# Align dates
common_idx_lo = strategy_excess_lo.index.intersection(bench_excess_lo.index)
strategy_excess_lo = strategy_excess_lo.reindex(common_idx_lo)
bench_excess_lo = bench_excess_lo.reindex(common_idx_lo)

qs.reports.html(
    strategy_excess_lo,
    benchmark=bench_excess_lo.to_frame("Market"),
    rf=0.0,
    periods_per_year=252,
    output="out/oos_long_only_tearsheet.html",
    title=f"OUT-OF-SAMPLE: Long-Only ({dates_outofsample.min().date()} to {dates_outofsample.max().date()})"
)
print("   ✓ Saved: out/oos_long_only_tearsheet.html")
print("   Strategy: Top 30% Long (100% invested)")
print(f"   Period: {strategy_excess_lo.index.min().date()} to {strategy_excess_lo.index.max().date()}")
print(f"   Days: {len(strategy_excess_lo)}")


# SUMMARY

print("\nHTML REPORTS GENERATED")

print("\n  Two OUT-OF-SAMPLE reports created:")
print("\n1.   Long-Short Market Neutral:")
print("   File: out/oos_long_short_tearsheet.html")
print(f"   Period: {dates_outofsample.min().date()} to {dates_outofsample.max().date()}")
print("   Allocation: Top 20% Long, Bottom 20% Short")
print("   Net Exposure: 0% (market neutral)")

print("\n2.   Long-Only:")
print("   File: out/oos_long_only_tearsheet.html")
print(f"   Period: {dates_outofsample.min().date()} to {dates_outofsample.max().date()}")
print("   Allocation: Top 30% Long")
print("   Net Exposure: 100% (directional)")


print("\n  Both reports show TRUE out-of-sample performance!")
print("  Cell 20 shows in-sample vs out-of-sample comparison.")



Generating Out-Of-Sample HTML Reports
   These reports show TRUE out-of-sample performance only!

1. Generating Out-Of-Sample Long-Short report...
   ✓ Saved: out/oos_long_short_tearsheet.html
   Strategy: Top 20% Long, Bottom 20% Short (Market Neutral)
   Period: 2020-06-15 to 2020-12-30
   Days: 139

2. Generating Out-Of-Sample Long-Only report...
   ✓ Saved: out/oos_long_only_tearsheet.html
   Strategy: Top 30% Long (100% invested)
   Period: 2020-06-15 to 2020-12-30
   Days: 139

HTML REPORTS GENERATED

  Two OUT-OF-SAMPLE reports created:

1.   Long-Short Market Neutral:
   File: out/oos_long_short_tearsheet.html
   Period: 2020-06-12 to 2020-12-30
   Allocation: Top 20% Long, Bottom 20% Short
   Net Exposure: 0% (market neutral)

2.   Long-Only:
   File: out/oos_long_only_tearsheet.html
   Period: 2020-06-12 to 2020-12-30
   Allocation: Top 30% Long
   Net Exposure: 100% (directional)

  Both reports show TRUE out-of-sample performance!
  Cell 20 shows in-sample vs out-of-sample

In [362]:
# Performance Metrics

# Classification accuracy
true_class_val = DIR_binary.loc[used_mask]
pred_class_val = pred_class.loc[used_mask]

cm = confusion_matrix(true_class_val, pred_class_val, labels=[0, 1])
report = classification_report(
    true_class_val, pred_class_val,
    labels=[0, 1],
    target_names=["Down", "Up"],
    zero_division=0
)

# Risk-adjusted returns
rf_series = df.loc[used_mask, "rf"].groupby(level="date").mean()
excess_ret = port_ret_by_date - rf_series

ann_factor = 252.0
mu_ann = excess_ret.mean() * ann_factor
sd_ann = excess_ret.std(ddof=0) * np.sqrt(ann_factor)
sharpe = mu_ann / sd_ann if sd_ann > 0 else np.nan

# Drawdown
roll_max = equity.cummax()
dd = 1.0 - equity / roll_max
max_dd = dd.max()


print("\nPerformance Summary")
print(f"Classification Accuracy: {(pred_class_val == true_class_val).mean():.1%}")
print("\nRisk-Adjusted Returns:")
print(f"  Annualized excess return: {mu_ann:.2%}")
print(f"  Annualized volatility: {sd_ann:.2%}")
print(f"  Sharpe ratio: {sharpe:.3f}")
print(f"  Maximum drawdown: {max_dd:.2%}")


Performance Summary
Classification Accuracy: 53.1%

Risk-Adjusted Returns:
  Annualized excess return: 2.88%
  Annualized volatility: 35.67%
  Sharpe ratio: 0.081
  Maximum drawdown: 34.60%


In [363]:
# Detailed Performance Report

print("\nValidation Coverage")
print(f"Validated dates: {port_ret_by_date.index.min().date()} to {port_ret_by_date.index.max().date()}")
print(f"Trading days: {len(port_ret_by_date)}")
print(f"Validated observations: {used_mask.sum():,} of {len(df):,}")


print("\nConfusion Matrix (Binary Classification)")
cm_df = pd.DataFrame(cm, index=["Actual Down", "Actual Up"], columns=["Pred Down", "Pred Up"])
print(cm_df)


print("\nClassification Report")
print(report)


print("\nPortfolio Statistics")
print(f"Annualized excess return: {mu_ann:.2%}")
print(f"Annualized volatility:    {sd_ann:.2%}")
print(f"Sharpe ratio (excess):    {sharpe:.3f}")
print(f"Maximum drawdown:         {max_dd:.2%}")

print("\nEquity Curve Sample")
print("First 5 days:")
print(equity.head())
print("\nLast 5 days:")
print(equity.tail())



Validation Coverage
Validated dates: 2019-07-25 to 2020-06-05
Trading days: 219
Validated observations: 17,301 of 54,984

Confusion Matrix (Binary Classification)
             Pred Down  Pred Up
Actual Down       2043     6148
Actual Up         1967     7143

Classification Report
              precision    recall  f1-score   support

        Down       0.51      0.25      0.33      8191
          Up       0.54      0.78      0.64      9110

    accuracy                           0.53     17301
   macro avg       0.52      0.52      0.49     17301
weighted avg       0.52      0.53      0.49     17301


Portfolio Statistics
Annualized excess return: 2.88%
Annualized volatility:    35.67%
Sharpe ratio (excess):    0.081
Maximum drawdown:         34.60%

Equity Curve Sample
First 5 days:
date
2019-07-25    1.0
2019-07-26    1.0
2019-07-29    1.0
2019-07-30    1.0
2019-07-31    1.0
dtype: float64

Last 5 days:
date
2020-06-01    1.000431
2020-06-02    1.010678
2020-06-03    1.010160
2020-